In [30]:
import pyhmmer
from Bio import SeqIO
from Bio.Seq import Seq
import pandas as pd
from pathlib import Path

In [32]:
hmm_dir = "hmms/"
contigs_fasta = "contigs.fa"

thresholds = {
    "Astroviridae_1A": 15,
    "Astroviridae_1B": 15,
    "Astroviridae_2": 15,
    "Astroviridae_1B_pcr_primer_region_RDRP": 15,
    "Kobuvirus_VP0-VP3-VP1": 15,
    "Kobuvirus_3A_3B_3C_3D": 15,
    "Kobuvirus_2A_2B_2C": 15,
    "Kobuvirus_polyprotein": 15,
    "Parechovirus_VP0_VP3_VP1": 12,
    "Parechovirus_2A_2B_2C": 12,
    "Parechovirus_3A_3B_3C_3D": 12,
    "Parechovirus_polyprotein": 12,
}

In [34]:
def norm(x):
    return x.decode() if isinstance(x, bytes) else str(x)

def load_hmms(hmm_dir):
    hmms = []
    for hmm_file in Path(hmm_dir).glob("*.hmm"):
        with pyhmmer.plan7.HMMFile(hmm_file) as f:
            for hmm in f:
                hmms.append(hmm)
    return hmms

hmms = load_hmms(hmm_dir)
print(f"Loaded {len(hmms)} HMMs")

def get_threshold(hmm):
    name = norm(hmm.name)
    return thresholds.get(name, 0.0)

Loaded 12 HMMs


In [35]:
def translate_six_frames(record, min_len=100):
    seq = record.seq
    results = []

    for strand, nuc in [(+1, seq), (-1, seq.reverse_complement())]:
        for frame in range(3):
            prot = nuc[frame:].translate(to_stop=False)

            if len(prot) < min_len:
                continue

            frame_id = frame + 1 if strand == 1 else -(frame + 1)

            results.append({
                "protein": str(prot),
                "frame": frame_id
            })

    return results

In [48]:
alphabet = pyhmmer.easel.Alphabet.amino()

def contigs_to_proteins(fasta):
    proteins = []
    contig_lengths = {}

    for record in SeqIO.parse(fasta, "fasta"):
        contig_lengths[record.id] = len(record.seq)

        frames = translate_six_frames(record)

        for f in frames:
            text_seq = pyhmmer.easel.TextSequence(
                name=f"{record.id}|frame={f['frame']}".encode(),
                sequence=f["protein"]
            )
            proteins.append(text_seq.digitize(alphabet))

    return proteins, contig_lengths


proteins, contig_lengths = contigs_to_proteins(contigs_fasta)

c:\Users\Mate_\miniconda3\envs\virome\Lib\site-packages\Bio\Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [49]:
results = []

for hmm in hmms:
    hmm_name = norm(hmm.name)
    threshold = get_threshold(hmm)

    for top_hits in pyhmmer.hmmer.hmmsearch(
        hmm,
        proteins,
        E=1000,
        domE=1000
    ):
        for hit in top_hits:
            hit_name = norm(hit.name)

            if not hit.domains:
                continue

            for domain in hit.domains:
                ali = domain.alignment

                hmm_len = hmm.M
                ali_len = ali.hmm_to - ali.hmm_from + 1

                coverage = ali_len / hmm_len

                contig = hit_name.split("|")[0]

                results.append({
                    "Query": hmm_name,
                    "Name": hit_name,
                    "Contig": contig,
                    "Frame": hit_name.split("frame=")[-1],
                    "Score": domain.score,
                    "From": ali.hmm_from,
                    "To": ali.hmm_to,
                    "Alignment_length": ali_len,
                    "HMM_length": hmm_len,
                    "Coverage": coverage,
                    "Threshold": threshold,
                    "Score_ratio": domain.score / threshold if threshold > 0 else None,
                    "Length_contig": contig_lengths.get(contig, None)
                })

In [50]:
df = pd.DataFrame(results)

df = df.sort_values("Score", ascending=False)

df

,Query,Name,Contig,Frame,Score,From,To,Alignment_length,HMM_length,Coverage,Threshold,Score_ratio,Length_contig
995,Parechovirus_polyprotein,k141_17975|frame=-2,k141_17975,-2,2433.652100,9,2204,2196,2205,0.995918,12,202.804342,7045
1263,Parechovirus_VP0_VP3_VP1,k141_17975|frame=-2,k141_17975,-2,866.410217,18,807,790,808,0.977723,12,72.200851,7045
996,Parechovirus_polyprotein,k141_16661|frame=-2,k141_16661,-2,847.427307,103,1399,1297,2205,0.588209,12,70.618942,6888
109,Astroviridae_1B,k141_18644|frame=-1,k141_18644,-1,839.591370,10,516,507,516,0.982558,15,55.972758,4882
0,Astroviridae_1A,k141_31770|frame=-2,k141_31770,-2,834.127136,26,830,805,858,0.938228,15,55.608476,6707
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1422,Parechovirus_2A_2B_2C,k141_10396|frame=1,k141_10396,1,-4.125596,376,414,39,594,0.065657,12,-0.343800,5580
883,Kobuvirus_3A_3B_3C_3D,k141_10396|frame=1,k141_10396,1,-4.133823,127,154,28,627,0.044657,15,-0.275588,5580
1093,Parechovirus_polyprotein,k141_25930|frame=1,k141_25930,1,-4.206866,630,685,56,2205,0.025397,12,-0.350572,6623
1319,Parechovirus_VP0_VP3_VP1,k141_15118|frame=-2,k141_15118,-2,-4.281809,679,728,50,808,0.061881,12,-0.356817,2119


In [51]:
df_filtered = df[
    (df["Score_ratio"] >= 3) &
    (df["Coverage"] >= 0.3)
]

In [52]:
df_filtered

,Query,Name,Contig,Frame,Score,From,To,Alignment_length,HMM_length,Coverage,Threshold,Score_ratio,Length_contig
995,Parechovirus_polyprotein,k141_17975|frame=-2,k141_17975,-2,2433.652100,9,2204,2196,2205,0.995918,12,202.804342,7045
1263,Parechovirus_VP0_VP3_VP1,k141_17975|frame=-2,k141_17975,-2,866.410217,18,807,790,808,0.977723,12,72.200851,7045
996,Parechovirus_polyprotein,k141_16661|frame=-2,k141_16661,-2,847.427307,103,1399,1297,2205,0.588209,12,70.618942,6888
109,Astroviridae_1B,k141_18644|frame=-1,k141_18644,-1,839.591370,10,516,507,516,0.982558,15,55.972758,4882
0,Astroviridae_1A,k141_31770|frame=-2,k141_31770,-2,834.127136,26,830,805,858,0.938228,15,55.608476,6707
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1420,Parechovirus_2A_2B_2C,k141_5211|frame=2,k141_5211,2,36.536781,146,416,271,594,0.456229,12,3.044732,5098
1540,Parechovirus_3A_3B_3C_3D,k141_18644|frame=-1,k141_18644,-1,36.287399,220,460,241,567,0.425044,12,3.023950,4882
1419,Parechovirus_2A_2B_2C,k141_25930|frame=1,k141_25930,1,36.209949,28,286,259,594,0.436027,12,3.017496,6623
1423,Parechovirus_2A_2B_2C,k141_10396|frame=1,k141_10396,1,36.101082,28,286,259,594,0.436027,12,3.008423,5580


In [53]:
df_best = (
    df_filtered
    .sort_values("Score", ascending=False)
    .groupby(["Contig", "Query"])
    .head(1)
)

In [54]:
df_top = (
    df_best
    .sort_values("Score", ascending=False)
    .groupby("Contig")
    .head(1)
)

In [55]:
df.to_csv("all_hits.csv", index=False)
df_filtered.to_csv("filtered_hits.csv", index=False)
df_top.to_csv("top_annotations.csv", index=False)